# 01b: Near-duplicates in EyePACS

A 64-bit perceptual hash only sees the coarse layout of a fundus photo, so different eyes often
share it. This notebook takes the pairs whose hashes are close (candidates) and compares their
fine structure (vessels, lesions) to decide which are real duplicates. The similarity threshold
is set from two reference distributions measured on the same images:
- **copies:** processed images re-encoded the way another dataset might have stored them
  (downscaled, JPEG, brighter, slightly re-cropped): these must count as duplicates;
- **different eyes:** random pairs from different patients: these must not.

**Settings:** Accelerator **None**, Internet **On**.
**Input:** *Add Input* → *Notebooks* → *Your Work* → **01: Preprocess EyePACS** (its saved output).
**Run:** *Save Version* → *Save & Run All (Commit)*. Takes about 15–30 min.

**Output** (`/kaggle/working/`): `duplicate_candidates.parquet`, `duplicates_summary.json`,
`similarity.png`, `top_pairs.png`.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
OUT_DIR = "/kaggle/working"
N_CONTROLS = 1000
SEED = 0

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Locate the processed images and manifest ------------------------------
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from retinaguard.data.processing import split_processed_path

INPUT_ROOT = Path("/kaggle/input")
found = sorted(INPUT_ROOT.glob("**/manifest_eyepacs.parquet"))
assert found, "manifest_eyepacs.parquet not found: attach the output of notebook 01 (Add Input)"
shard_dir = found[0].parent
manifest = pd.read_parquet(found[0])
ok = manifest.dropna(subset=["processed_path"]).reset_index(drop=True)
print(f"{len(ok):,} processed images in {shard_dir}")
out_dir = Path(OUT_DIR)
rng = np.random.default_rng(SEED)

In [ ]:
# --- 2. Candidate pairs: phashes within the Hamming distance ------------------
from retinaguard.data.duplicates import CANDIDATE_MAX_DISTANCE, candidate_pairs, hash_values

t0 = time.perf_counter()
pairs = candidate_pairs(ok["phash"].tolist())
values = hash_values(ok["phash"].tolist())
cand = pd.DataFrame({"a": pairs[:, 0], "b": pairs[:, 1]})
cand["phash_distance"] = [int(values[a] ^ values[b]).bit_count() for a, b in pairs]
patients = ok["patient_id"].to_numpy()
cand["same_patient"] = patients[pairs[:, 0]] == patients[pairs[:, 1]]
print(
    f"{len(cand):,} candidate pairs (distance <= {CANDIDATE_MAX_DISTANCE}) "
    f"in {time.perf_counter() - t0:.0f}s"
)
print("by distance:", cand["phash_distance"].value_counts().sort_index().to_dict())
print(f"same patient: {int(cand['same_patient'].sum())}")

In [ ]:
# --- 3. Structure signatures for every candidate image + control samples -----
import cv2

from retinaguard.data.duplicates import structure_signature
from retinaguard.data.preprocess import decode_rgb, encode_jpeg, preprocess, resize_square

control_idx = rng.choice(len(ok), size=N_CONTROLS, replace=False)
wanted = sorted(set(cand["a"]) | set(cand["b"]) | set(control_idx.tolist()))


def reframed(img):
    """Same photo with a different black frame and resolution, cropped again by our pipeline."""
    padded = np.pad(img, ((10, 30), (40, 5), (0, 0)))
    small = cv2.resize(padded, None, fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
    return preprocess(small, img.shape[0])[0]


def clipped(img):
    """Same photo with the top and bottom of the disc cut off by the camera frame."""
    out = img.copy()
    k = round(0.04 * img.shape[0])
    out[:k] = 0
    out[-k:] = 0
    return preprocess(out, img.shape[0])[0]


def copies(img):
    """Ways another dataset may have stored the same photo."""
    size = img.shape[0]
    return {
        "downscaled_256": resize_square(resize_square(img, 256), size),
        "jpeg_75": decode_rgb(encode_jpeg(img, quality=75)),
        "brighter_10pct": np.clip(img.astype(np.float32) * 1.1, 0, 255).astype(np.uint8),
        "reframed": reframed(img),
        "clipped_4pct": clipped(img),
        "all_combined": decode_rgb(encode_jpeg(resize_square(reframed(img), 256), quality=75)),
    }


from retinaguard.data.hashing import hamming, phash

signatures = {}
copy_rows = []
by_shard = ok.loc[wanted, "processed_path"].map(split_processed_path)
t0 = time.perf_counter()
control_set = set(control_idx.tolist())
for shard_file, members in by_shard.groupby(by_shard.str[0]):
    with zipfile.ZipFile(shard_dir / shard_file) as archive:
        for idx, (_, member) in members.items():
            img = decode_rgb(archive.read(member))
            signatures[idx] = structure_signature(img)
            if idx in control_set:
                for kind, copy in copies(img).items():
                    copy_rows.append(
                        {
                            "idx": idx,
                            "kind": kind,
                            "similarity": float(signatures[idx] @ structure_signature(copy)),
                            "phash_distance": hamming(ok.at[idx, "phash"], phash(copy)),
                        }
                    )
    print(
        f"{shard_file}: {len(signatures):,}/{len(wanted):,} signatures, "
        f"{time.perf_counter() - t0:.0f}s",
        flush=True,
    )
copies_df = pd.DataFrame(copy_rows)

In [ ]:
# --- 4. Similarities: copies vs different eyes vs candidates ------------------
a, b = rng.choice(control_idx, size=(2, 5 * N_CONTROLS))
keep = patients[a] != patients[b]


def similarity(i, j):
    return float(signatures[i] @ signatures[j])


different = np.array([similarity(i, j) for i, j in zip(a[keep], b[keep], strict=True)])
cand["similarity"] = [similarity(i, j) for i, j in zip(cand["a"], cand["b"], strict=True)]


def q(x):
    return [round(float(v), 3) for v in np.quantile(x, [0, 0.01, 0.5, 0.99, 1])] if len(x) else []


print("quantiles  min / p01 / p50 / p99 / max")
for kind, grp in copies_df.groupby("kind"):
    within = float((grp["phash_distance"] <= CANDIDATE_MAX_DISTANCE).mean())
    print(f"copy {kind:<16} {q(grp['similarity'])}  phash within candidate distance: {within:.1%}")
print(f"different eyes       {q(different)}")
print(f"candidate pairs      {q(cand['similarity'])}")
for t in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    print(
        f"threshold {t}: copies kept {float((copies_df['similarity'] >= t).mean()):.2%}, "
        f"different eyes passing {float((different >= t).mean()):.2%}, "
        f"candidates passing {int((cand['similarity'] >= t).sum())}"
    )

In [ ]:
# --- 5. Plot: one panel per distribution, shared similarity axis -------------
import matplotlib.pyplot as plt

from retinaguard.data.duplicates import DUPLICATE_MIN_SIMILARITY

panels = [
    ("Re-encoded copies (must be duplicates)", copies_df["similarity"]),
    ("Different patients (must not be)", different),
    ("Candidate pairs (phash close)", cand["similarity"]),
]
bins = np.linspace(-0.2, 1, 61)
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
for ax, (title, x) in zip(axes, panels, strict=True):
    ax.hist(x, bins=bins, color="#3b6ea8", edgecolor="white", linewidth=0.5)
    ax.axvline(DUPLICATE_MIN_SIMILARITY, color="#444444", linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    ax.set_title(f"{title}: n = {len(x):,}", fontsize=10, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylabel("pairs (log)")
axes[-1].set_xlabel(f"structure similarity (dashed: current threshold {DUPLICATE_MIN_SIMILARITY})")
plt.tight_layout()
fig.savefig(out_dir / "similarity.png", dpi=90)
plt.show()

In [ ]:
# --- 6. Look at the most similar candidate pairs -----------------------------
def load(idx):
    shard_file, member = split_processed_path(ok.at[idx, "processed_path"])
    with zipfile.ZipFile(shard_dir / shard_file) as archive:
        return decode_rgb(archive.read(member))


top = cand.sort_values("similarity", ascending=False).head(8)
if len(top):
    fig, axes = plt.subplots(len(top), 2, figsize=(6, 3 * len(top)), squeeze=False)
    for row, (_, pair) in zip(axes, top.iterrows(), strict=True):
        for ax, idx in zip(row, (pair["a"], pair["b"]), strict=True):
            ax.imshow(load(idx))
            ax.set_title(f"{ok.at[idx, 'image_id']}  grade {ok.at[idx, 'dr_grade']}", fontsize=8)
            ax.axis("off")
        row[0].text(
            0,
            -0.12,
            f"similarity {pair['similarity']:.3f}, phash distance {pair['phash_distance']}",
            transform=row[0].transAxes,
            fontsize=8,
        )
    plt.tight_layout()
    fig.savefig(out_dir / "top_pairs.png", dpi=80)
    plt.show()

In [ ]:
# --- 7. Save -------------------------------------------------------------------
import json

cand["image_id_a"] = ok["image_id"].to_numpy()[cand["a"]]
cand["image_id_b"] = ok["image_id"].to_numpy()[cand["b"]]
cand[["image_id_a", "image_id_b", "phash_distance", "same_patient", "similarity"]].to_parquet(
    out_dir / "duplicate_candidates.parquet", index=False
)
summary = {
    "candidate_pairs": len(cand),
    "candidates_by_distance": cand["phash_distance"].value_counts().sort_index().to_dict(),
    "copies": {k: q(g["similarity"]) for k, g in copies_df.groupby("kind")},
    "copies_phash_within_candidate_distance": {
        k: round(float((g["phash_distance"] <= CANDIDATE_MAX_DISTANCE).mean()), 4)
        for k, g in copies_df.groupby("kind")
    },
    "different_eyes": q(different),
    "candidates": q(cand["similarity"]),
    "candidates_at_or_above": {
        str(t): int((cand["similarity"] >= t).sum()) for t in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
    },
}
(out_dir / "duplicates_summary.json").write_text(json.dumps(summary, indent=2, default=int))
print(json.dumps(summary, indent=2, default=int))